# Exploratory Data Analysis

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os
import re

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

In [ ]:
pd.options.mode.copy_on_write = True

In [ ]:
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# Remake into string variable
items_tagged['dish_category'] = items_tagged['dish_category'].astype(str)

# How many unique items must it have to be considered a useful category?
unique_items = 3
dish_category_counts = items_tagged['dish_category'].value_counts().nlargest(20)
usable_categories = dish_category_counts[dish_category_counts > unique_items].index.tolist()

# Throw these smaller categories into a larger 'Other' category
dish_categories_other = items_tagged['dish_category'].where(items_tagged['dish_category'].isin(usable_categories), 'Other')
items_tagged = items_tagged.assign(dish_category=dish_categories_other)

# Add alcohol column
items_tagged['is_alcohol'] = np.where(items_tagged['dish_category'] == 'Alcohol', 'yes', 'no')

## Static Reference Data Exploration

### Menu Stats

In [ ]:
print(f'Total number of menu items: {items_tagged.shape[0]}')
print(f'Median number of menu items for a restaurant: {round(items_tagged.groupby("location_id")["item_name"].count().median())}')

### Menu Visuals

In [ ]:
# How many categories to include?
topn = 20

## Colors for the bars
colors1 = ["#ef8a62", "#67a9cf"]
colors2 = ["#ff6961", "#aec6cf", "#77dd77"]

# Calculate the counts for each combination
pivot_df1 = items_tagged.groupby(['is_plant_based', 'is_alcohol'], observed=True).size().unstack(fill_value=0)
pivot_df1 = pivot_df1.loc[pivot_df1.sum(axis=1).sort_values(ascending=False).index] # Sort

# Calculate the counts for each combination
pivot_df2 = items_tagged.groupby(['item_type', 'is_plant_based'], observed=True).size().unstack(fill_value=0)
pivot_df2 = pivot_df2.loc[pivot_df2.sum(axis=1).sort_values(ascending=False).index] # Sort

# Calculate the counts for each combination
pivot_df3 = items_tagged.groupby(['dish_category', 'is_plant_based'], observed=True).size().unstack(fill_value=0)
pivot_df3 = pivot_df3.loc[pivot_df3.sum(axis=1).sort_values(ascending=False).index[:topn]] # Sort

# Calculate the counts for each combination
pivot_df4 = items_tagged.groupby(['dish_category', 'is_plant_based'], observed=True).size().unstack(fill_value=0)
pivot_df4 = pivot_df4.loc[pivot_df4.sum(axis=1).sort_values(ascending=False).index[3:topn]] # Sort

# Add to lists
dfs = [pivot_df1, pivot_df2, pivot_df3, pivot_df4]
color_groups = [colors1, colors2, colors2, colors2]
titles = ["Plant Based", "Meal Type Counts", "Dish Category Counts", "Dish Category Counts w/o Alcohol or Unknowns"]
xlabs = ["Plant Based", "Meal Type", "Dish Category", "Dish Category"]
legends = ["Is Alcohol?", "Is Plant Based?", "Is Plant Based?", "Is Plant Based?"]
coordinates = [(0,0),(0,1),(1,0),(1,1)]

# Create a 2x2 grid of subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Zip together for looping
data_for_visual = zip(dfs, color_groups, titles, xlabs, legends, coordinates)
for i, data_tuple in enumerate(data_for_visual):

    # Unpack
    df, colors, title, xlab, legend, coordinate = data_tuple
    a, b = coordinate

    # Initialize a series of zeros with the same index
    bottom = pd.Series(0, index=df.index)

    # Loop through every variable to be stacked
    for j, col in enumerate(df.columns):

        # Stack up multiple bar charts
        sns.barplot(x=df.index.tolist(), 
                    y=df[col], 
                    bottom=bottom, 
                    color=colors[j], 
                    label=col,
                    ax=axes[a,b])
        
        # Start the next level exactly where the last one
        bottom = bottom + df[col]

    axes[a,b].set_ylabel('Count')
    axes[a,b].set_xlabel(xlab)
    axes[a,b].set_xticks(df.index)
    axes[a,b].set_xticklabels(labels=df.index.tolist(), rotation=70)
    axes[a,b].set_title(title)
    axes[a,b].legend(title=legend)

# Adjust layout
plt.tight_layout()

# Show the plots
plt.show()

### Customer Stats

Merge in customer data

In [ ]:
# Initialize dict all data
sales_and_customers_data = {}
for loc_id, df in sales_and_menu_data.items():

    # Prevent overwriting
    df = df.copy()

    # Keep the index, since merges don't keep it
    df.reset_index(inplace=True)

    # Double check customers are unique
    customers.dropna(subset=['customer_id'], inplace=True)
    customers.drop_duplicates(subset=['location_id', 'customer_id'], inplace=True)

    # Combine
    merged = pd.merge(df, customers, on=['location_id', 'customer_id'], how='left')

    # Reset the index back to datetimes
    merged.set_index('created_at', inplace=True, drop=False)

    # Save
    sales_menu_customers_data[loc_id] = merged

Totals

In [ ]:
# Number of customers, number of customers with gender data, number of customers with age data
print(f'Total number of customers: {customers.shape[0]:,}')
print(f'Customers with gender data: {customers["gender"].notna().sum():,}')
print(f'Customers with age data: {customers["age"].notna().sum():,}')

Customers that went to multiple restaurants

In [ ]:
# Precompute customers for efficiency
precomputed_customers = {}
for loc_id in location_ids:
    precomputed_customers[loc_id] = set(fdf(customers).filter('location_id', loc_id)['customer_id'].unique().tolist())

for i in range(len(location_ids)):
    loc_id1 = location_ids[i]
    customer_set1 = precomputed_customers[loc_id1]
    for loc_id2 in location_ids[i:]:
        if loc_id1 != loc_id2:
            customer_set2 = precomputed_customers[loc_id2]
            intersection = customer_set1.intersection(customer_set2)
            if intersection:
                print(loc_id1, loc_id2, len(intersection))
            else:
                pass
                # print(loc_id1, loc_id2, "Nope")

Female proportions

In [ ]:
female_proportions_list = []

for loc_id in location_ids:

    row = {'location_id': loc_id}

    cross_over = before_after_details.loc[loc_id]['cross_over_date'].tz_localize(None)

    df = sales_menu_customers_data[loc_id].copy()
    df.index = df.index.tz_localize(None)

    before_genders = df.loc[:cross_over]['gender'].value_counts()
    after_genders = df.loc[cross_over:]['gender'].value_counts()
    
    if not before_genders.empty and not after_genders.empty:

        before_female_total = before_genders.loc['female']
        after_female_total = after_genders.loc['female']
        before_known_gender_total = before_genders.loc['male'] + before_genders.loc['female']
        after_known_gender_total = after_genders.loc['male'] + after_genders.loc['female']

        before_frac_female = -1
        if before_known_gender_total != 0:
            before_frac_female = before_female_total/before_known_gender_total
            
        after_frac_female = -1
        if after_known_gender_total != 0:
            after_frac_female = after_female_total/after_known_gender_total
            
        sample_qualifer = ""
        if 1000 < before_known_gender_total and 1000 < after_known_gender_total:
            sample_qualifer = "Big Enough Sample"

        row['female_proportion_before'] = round(before_frac_female*100)/100
        row['female_proportion_after'] = round(after_frac_female*100)/100
        row['enough_data'] = bool(sample_qualifer)
        row['before_known_gender_total'] = before_known_gender_total
        row['after_known_gender_total'] = after_known_gender_total

    elif before_genders.empty and after_genders.empty:

        print(loc_id, "--No customer data!")

    else:

        print(loc_id, "--Not enough data before or after.")

    female_proportions_list.append(row)

female_proportions = pd.DataFrame(female_proportions_list)

View

In [ ]:
female_proportions

Revisiting customers

In [ ]:
customer_revisit_row_list = []
for loc_id, df in sales_and_menu_data.items():
    
    df = df.copy()
    df.reset_index(inplace=True)

    row = {'location_id': loc_id, 'total': df['customer_id'].nunique()}

    for j in [1, 2, 5, 10]:

        customers_revisits_j = df.groupby('customer_id', observed=True)['created_at'].nunique() > j
        num_customer_revisits = customers_revisits_j.sum()
        row['more than ' + str(j)] = num_customer_revisits

    customer_revisit_row_list.append(row)

revisits = pd.DataFrame(customer_revisit_row_list)

View

In [ ]:
revisits

## Restaurant Sales Data Exploration

Totals

In [ ]:
# Recheck total number of entries
sales_items_total = 0
sales_data_total = 0
sales_transactions_total = 0
for loc_id, df in sales_and_menu_data.items():
    sales_items_total += df['item_quantity'].sum()
    sales_data_total += df.shape[0]
    sales_transactions_total += df['order_id'].nunique()

# Print
print(f'Total number of items sold: {sales_items_total:,}')
print(f'Total number of sales entries (row): {sales_data_total:,}')
print(f'Total number of transactions: {sales_transactions_total:,}')

Timeframes

In [ ]:
list_of_timeframes = []

# Find the time difference
for location_id, df in tqdm(sales_and_menu_data.items()):

    # Find the time difference
    timedelta = df.index[-1] - df.index[0]

    # Convert to days, then to years
    years = timedelta.days / 365.25

    # Append
    list_of_timeframes.append(years)

# Turn into an np array for finding median, mean, and std
timeframes = np.array(list_of_timeframes)

# Display
print("Median: {:.2f} year range".format(np.median(timeframes)))
print("Mean: {:.2f} year range".format(np.mean(timeframes)))
print("SD: {:.2f} years".format(np.std(timeframes)))
print("Restaurants with less than a 2 year range: {}".format((timeframes < 2).sum()))

### Promotional Items

In [ ]:
promo_match_list = []

for loc_id, df in merged_sales_and_menu.items():

    # Looking for promo
    promo_item = before_after_details.loc[loc_id, 'first_plant_based_mention']
    cross_over_date = before_after_details.loc[loc_id, 'cross_over_date']
    promo_df = fdf(df).filter('item_name', promo_item)
    ever_found = not promo_df.empty


    # Actual first
    plant_based = fdf(df).filter('is_plant_based', 'yes')
    first_plant_based = plant_based['item_name'].iloc[0]
    its_date = plant_based.index[0]

    # Do they match?
    is_first = promo_item.lower() == first_plant_based.lower()

    row = {'location_id': loc_id, 'promo_item': promo_item, 'cross_over_date': cross_over_date, 'ever_found': ever_found, 'is_first': is_first, 'first_plant_based': first_plant_based, 'its_date': its_date}

    promo_match_list.append(row)
    
pd.DataFrame(promo_match_list)

### Sales Visuals

In [ ]:
num_plots = len(sales_and_menu_data)
cols = 4 
rows = math.ceil(num_plots / cols) * 4

# Create a figure with multiple subplots
fig, axs = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axs = axs.flatten()  # Flatten the array for easy indexing

top_dishes_number = 20

for i, (location_id, data) in tqdm(enumerate(sales_and_menu_data.items())):

    df = data.copy()
    
    # Get the top 20 items sorted in descending order
    top_items_by_times_ordered = df['item_name'].value_counts().nlargest(top_dishes_number).sort_values()

    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i].barh(top_items_by_times_ordered.index.str.slice(0,20).str.capitalize(), top_items_by_times_ordered)
    axs[4*i].set_title(f'{location_id}\nTotal by Times Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i].tick_params(axis='y', labelsize=10)
    axs[4*i].set_xlabel('Times Ordered')

    top_items_by_quantity_ordered = df.groupby('item_name')['item_quantity'].sum().nlargest(top_dishes_number).sort_values()
    
    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i + 1].barh(top_items_by_quantity_ordered.index.str.slice(0,20).str.capitalize(), top_items_by_quantity_ordered, color='cyan')
    axs[4*i + 1].set_title(f'{location_id}\nTotal by Quantity Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i + 1].tick_params(axis='y', labelsize=10)
    axs[4*i + 1].set_xlabel('Quantity Ordered')

    plant_df = df[df['is_plant_based'] == 'yes']
    
    # Get the top 20 items sorted in descending order
    top_plant_based_items_by_times_ordered = plant_df['item_name'].value_counts().nlargest(top_dishes_number).sort_values()

    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i + 2].barh(top_plant_based_items_by_times_ordered.index.str.slice(0,20).str.capitalize(), top_plant_based_items_by_times_ordered, color='green')
    axs[4*i + 2].set_title(f'{location_id}\nPlant-Based by Times Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i + 2].tick_params(axis='y', labelsize=10)
    axs[4*i + 2].set_xlabel('Times Ordered')

    top_plant_based_items_by_quantity_ordered = plant_df.groupby('item_name')['item_quantity'].sum().nlargest(top_dishes_number).sort_values()
    
    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i + 3].barh(top_plant_based_items_by_quantity_ordered.index.str.slice(0,20).str.capitalize(), top_plant_based_items_by_quantity_ordered, color='lime')
    axs[4*i + 3].set_title(f'{location_id}\nPlant-Based by Quantity Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i + 3].tick_params(axis='y', labelsize=10)
    axs[4*i + 3].set_xlabel('Quantity Ordered')

# # Add a global title at the top of the figure
fig.suptitle('Distribution of Plant-Based and Total Item Orders in Each Restaurant', fontsize=16)

# # Adjust layout and save the figure
plt.tight_layout()
plt.subplots_adjust(top=0.97)  # Adjust the top margin to make room for the global title
# plt.savefig('Restaurant Both Sales Distributions.png', bbox_inches='tight')